In [2]:
import os
import time
import numpy as np
from functools import lru_cache
from pathlib import Path

In [3]:
def ler_instancia(path):
    """
    Ordem exata:
    1: N
    2: FR (setup reman)
    3: F  (setup novo)
    4: HR (holding reman)
    5: H  (holding novo)
    6..6+N-1: D (N valores)
    6+N..6+2N-1: R (N valores)
    Linhas vazias e que começam com '#' são ignoradas.
    """
    tokens = []
    with open(path, 'r') as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith('#'):
                continue
            tokens += s.split()

    if len(tokens) < 5:
        raise ValueError("Arquivo inválido / muito curto.")

    idx = 0
    N  = int(float(tokens[idx])); idx += 1
    FR = float(tokens[idx]); idx += 1
    F  = float(tokens[idx]); idx += 1
    HR = float(tokens[idx]); idx += 1
    H  = float(tokens[idx]); idx += 1

    if len(tokens) < 5 + 2 * N:
        raise ValueError(f"Esperado D (N) e R (N) valores. Encontrados {len(tokens)-5}.")

    D = np.array([int(round(float(tokens[idx + i]))) for i in range(N)], dtype=int)
    idx += N
    R = np.array([int(round(float(tokens[idx + i]))) for i in range(N)], dtype=int)

    return N, FR, F, HR, H, D, R

In [4]:
def precompute_sums(D, R):
    N = len(D)
    sum_future_D = np.zeros(N+1, dtype=int)  # sum_future_D[t] = sum_{j=t}^{N-1} D[j]
    for t in range(N-1, -1, -1):
        sum_future_D[t] = sum_future_D[t+1] + D[t]
    sum_past_R = np.zeros(N, dtype=int)      # sum_past_R[t] = sum_{i=0}^{t} R[i]
    running = 0
    for t in range(N):
        running += R[t]
        sum_past_R[t] = running
    return sum_future_D, sum_past_R

In [5]:
result_path = Path('result/')

# DP (equivalente ao MIP )
def dp_exact(F, FR, H, HR, D, R, N_use):
    N = len(D)
    sum_future_D, sum_past_R = precompute_sums(D, R)

    @lru_cache(maxsize=None)
    def V(t, s, sr):
        # estado inválido
        if s < 0 or sr < 0:
            return float('inf')
        # terminal
        if t == N_use:
            return 0.0 if (s == 0 and sr == 0) else float('inf')

        best = float('inf')

        # limites teóricos
        Xmax = sum_future_D[t]
        XRmax_theoretical = min(sum_past_R[t], sum_future_D[t])

        demand_t = D[t]
        avail_reman_inventory = sr + R[t]   # o que há fisicamente disponível em t

        # escolher horizonte k (produzir em t para cobrir t..k)
        for k in range(t, N_use):
            # total necessário (produção nova + reman) para garantir que demandas t..k sejam cobertas
            total_needed = sum(D[t:k+1]) - s
            if total_needed < 0:
                total_needed = 0
            # teoricamente deve respeitar Xmax
            if total_needed > Xmax:
                continue

            # limite para x_r (remanufaturar em t):

            max_xr = min(total_needed, avail_reman_inventory, XRmax_theoretical)

            # enumerar x_r exaustivamente 0..max_xr (exato)
            for x_r in range(0, max_xr + 1):
                x_new = total_needed - x_r

                s_next = s + x_new + x_r - D[t]
                sr_next = sr + R[t] - x_r

                if s_next < 0 or sr_next < 0:
                    continue

                setup = (F if x_new > 0 else 0.0) + (FR if x_r > 0 else 0.0)
                immediate = setup + H * s_next + HR * sr_next

                fut = V(t + 1, s_next, sr_next)
                total = immediate + fut
                if total < best:
                    best = total

        #print(best)
        return best

    # reconstrução forward
    policy_x = [0] * N_use
    policy_xr = [0] * N_use
    s = 0; sr = 0
    for t in range(0, N_use):
        best = float('inf')
        best_choice = (0, 0, 0)  # (k, x_new, x_r)

        Xmax = sum_future_D[t]
        XRmax_theoretical = min(sum_past_R[t], sum_future_D[t])
        avail_reman_inventory = sr + R[t]

        for k in range(t, N_use):
            total_needed = sum(D[t:k+1]) - s
            if total_needed < 0:
                total_needed = 0
            if total_needed > Xmax:
                continue
            max_xr = min(total_needed, avail_reman_inventory, XRmax_theoretical)
            for x_r in range(0, max_xr + 1):
                x_new = total_needed - x_r
                s_next = s + x_new + x_r - D[t]
                sr_next = sr + R[t] - x_r
                if s_next < 0 or sr_next < 0:
                    continue
                setup = (F if x_new > 0 else 0.0) + (FR if x_r > 0 else 0.0)
                immediate = setup + H * s_next + HR * sr_next
                fut = V(t + 1, s_next, sr_next)
                total = immediate + fut
                if total < best:
                    best = total
                    best_choice = (k, x_new, x_r)

        k, x_new, x_r = best_choice
        policy_x[t] = int(x_new)
        policy_xr[t] = int(x_r)
        s = s + x_new + x_r - D[t]
        sr = sr + R[t] - x_r

    val = V(0, 0, 0)
    
    return val, policy_x, policy_xr

In [6]:
def main(path, N_use=None, arquivo=None):

    N, FR, F, HR, H, D, R = ler_instancia(path)

    if N_use is None:
        N_use = N

    N_use = min(N_use, N)

    start_time = time.time()
    total, x, x_r = dp_exact(F, FR, H, HR, D, R, N_use)
    total_rtime = time.time() - start_time
    
    arquivo.write(path+';'
                  +str(N_use)+';'
	 	+str(round(total,2))+';'
	 	+str(round(total_rtime,2))
	 	+'\n')

    s_hist = [0] * (N_use + 1)
    sr_hist = [0] * (N_use + 1)

    current_s = 0
    current_sr = 0

    for t in range(N_use):
        s_hist[t] = current_s
        sr_hist[t] = current_sr

        current_s = current_s + x[t] + x_r[t] - D[t]
        current_sr = current_sr + R[t] - x_r[t]

    s_hist[N_use] = current_s
    sr_hist[N_use] = current_sr

    #print("\n=========== RESULTADO ===========")
    #print(f"Instância: {path}")
    #print(f"Períodos usados: {N_use}")
    #print(f"Custo ótimo: {total:.4f}\n")

    #print("t | x_t (produção nova) | x_t_r (reman) | s_t | s_t_r")
    #print("-----------------------------------------------")
    #for t in range(N_use):
    #    print(f"{t+1:2d} | {x[t]:12d}        | {x_r[t]:7d}      | {s_hist[t+1]:3d} | {sr_hist[t+1]:3d}")

    #print("================================\n")

In [7]:
# Caminho

arquivo = open(os.path.join(result_path,'ulsr_dynamicp.txt'),'a')
for it in range(1,2):
    path = f"../../data/sifaleras/52_{it}.txt"
    main(path, N_use=10, arquivo=arquivo)
arquivo.close()
